In [1]:
# 03_sentiment.ipynb
# 목적: NewsAPI로 실제 뉴스 수집 + FinBERT 감성 분석

import requests
import pandas as pd
import os
from dotenv import load_dotenv
from transformers import pipeline

# 환경변수 로드
env_path = os.path.expanduser("~/Desktop/SentriVaR-500/.env")
load_dotenv(env_path)
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

# FinBERT 로드
nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

print("로드 완료")
print(f"API 키 확인: {NEWSAPI_KEY[:8]}...")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

로드 완료
API 키 확인: 243b95d4...


In [2]:
# NewsAPI로 실제 뉴스 헤드라인 수집 함수
def fetch_news(ticker, api_key, days_back=30):
    """
    특정 종목 관련 뉴스 헤드라인 수집
    - ticker: 종목명 (예: "Apple")
    - days_back: 며칠치 뉴스 가져올지
    """
    from datetime import datetime, timedelta

    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days_back)).strftime("%Y-%m-%d")

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": ticker,
        "from": start_date,
        "to": end_date,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 50,
        "apiKey": api_key
    }

    response = requests.get(url, params=params)
    data = response.json()

    if data["status"] != "ok":
        print(f"에러: {data.get('message')}")
        return []

    headlines = [article["title"] for article in data["articles"]]
    print(f"{ticker} 뉴스 {len(headlines)}개 수집 완료")
    return headlines

# 테스트
apple_news = fetch_news("Apple stock", NEWSAPI_KEY, days_back=30)
print("\n최근 헤드라인 3개:")
for h in apple_news[:3]:
    print(f"  - {h}")

Apple stock 뉴스 49개 수집 완료

최근 헤드라인 3개:
  - Asian markets retreat after rebounding AI stocks send the S&P 500 to brink of a new record
  - USMNT CRASHES OUT! USA 1-4 Belgium Immediate Live Reaction & Post-Mortem with Taylor Twellman
  - “Brands recognize the cultural dominance of podcasts”: global podcaster Mel Robbins talks AI, ad budgets and audience ownership


In [6]:
# 종목별 검색어 매핑 (더 정확한 검색을 위해)
TICKER_QUERIES = {
    "AAPL": "Apple Inc stock market",
    "MSFT": "Microsoft stock market",
    "GOOGL": "Google Alphabet stock market",
    "JPM": "JPMorgan Chase stock market",
    "SOXX": "semiconductor ETF iShares stock market"
}

# 전체 종목 뉴스 수집
all_news = {}
for ticker, query in TICKER_QUERIES.items():
    headlines = fetch_news(query, NEWSAPI_KEY, days_back=30)
    all_news[ticker] = headlines

print("\n수집 완료:")
for ticker, headlines in all_news.items():
    print(f"  {ticker}: {len(headlines)}개")

Apple Inc stock market 뉴스 49개 수집 완료
Microsoft stock market 뉴스 50개 수집 완료
Google Alphabet stock market 뉴스 50개 수집 완료
JPMorgan Chase stock market 뉴스 49개 수집 완료
semiconductor ETF iShares stock market 뉴스 49개 수집 완료

수집 완료:
  AAPL: 49개
  MSFT: 50개
  GOOGL: 50개
  JPM: 49개
  SOXX: 49개


In [7]:
# FinBERT로 감성 분석
def analyze_sentiment(headlines):
    """
    헤드라인 리스트를 받아서 감성 점수 반환
    - positive: +1, negative: -1, neutral: 0 으로 변환
    - 최종 점수: -1 ~ +1 사이
    """
    if not headlines:
        return 0.0

    score_map = {"positive": 1, "negative": -1, "neutral": 0}
    scores = []

    for headline in headlines:
        result = nlp(headline[:512])[0]  # 512자 제한
        score = score_map[result["label"]] * result["score"]
        scores.append(score)

    return round(sum(scores) / len(scores), 4)

# 전체 종목 감성 점수 계산
print("감성 분석 중... (1~2분 소요)")
sentiment_scores = {}
for ticker, headlines in all_news.items():
    sentiment_scores[ticker] = analyze_sentiment(headlines)
    print(f"  {ticker}: {sentiment_scores[ticker]}")

print("\n완료!")

감성 분석 중... (1~2분 소요)
  AAPL: -0.13
  MSFT: -0.0716
  GOOGL: -0.1526
  JPM: -0.1228
  SOXX: -0.1146

완료!


In [8]:
# 결과를 DataFrame으로 정리 후 저장
data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")
from datetime import datetime

sentiment_df = pd.DataFrame([{
    "date": datetime.today().strftime("%Y-%m-%d"),
    "ticker": ticker,
    "sentiment_score": score
} for ticker, score in sentiment_scores.items()])

sentiment_df.to_csv(f"{data_path}/sentiment_scores.csv", index=False)

print("저장 완료: sentiment_scores.csv")
print(sentiment_df)

저장 완료: sentiment_scores.csv
         date ticker  sentiment_score
0  2026-07-08   AAPL          -0.1300
1  2026-07-08   MSFT          -0.0716
2  2026-07-08  GOOGL          -0.1526
3  2026-07-08    JPM          -0.1228
4  2026-07-08   SOXX          -0.1146


In [16]:
# 뉴스 이원화 키워드 분류기
SYSTEMIC_KEYWORDS = [
    "federal reserve", "fed", "interest rate", "inflation", "gdp",
    "recession", "tariff", "trade war", "unemployment", "central bank",
    "treasury", "yield", "macro", "economy", "market crash"
]

IDIOSYNCRATIC_KEYWORDS = {
    "AAPL":  ["apple", "iphone", "tim cook", "app store", "mac"],
    "MSFT":  ["microsoft", "azure", "satya nadella", "windows", "copilot"],
    "GOOGL": ["google", "alphabet", "youtube", "sundar pichai", "gemini"],
    "JPM":   ["jpmorgan", "jamie dimon", "chase", "investment bank"],
    "SOXX":  ["semiconductor", "nvidia", "amd", "intel", "chips act", "tsmc"]
}

def classify_news(headline):
    """
    헤드라인을 Systemic / Idiosyncratic / Unrelated 로 분류
    """
    headline_lower = headline.lower()

    # 거시경제 뉴스 체크
    if any(kw in headline_lower for kw in SYSTEMIC_KEYWORDS):
        return "systemic"

    # 기업 고유 뉴스 체크
    for ticker, keywords in IDIOSYNCRATIC_KEYWORDS.items():
        if any(kw in headline_lower for kw in keywords):
            return f"idiosyncratic_{ticker}"

    return "unrelated"

# 테스트
test_headlines = [
    "Federal Reserve signals interest rate cuts in 2026",
    "Apple reports record iPhone sales in Q2",
    "JPMorgan warns of recession risk amid tariff uncertainty",
    "Google launches new Gemini AI model",
    "Local restaurant opens in downtown Manhattan"
]

for h in test_headlines:
    print(f"{classify_news(h):25s} → {h}")

systemic                  → Federal Reserve signals interest rate cuts in 2026
idiosyncratic_AAPL        → Apple reports record iPhone sales in Q2
systemic                  → JPMorgan warns of recession risk amid tariff uncertainty
idiosyncratic_GOOGL       → Google launches new Gemini AI model
unrelated                 → Local restaurant opens in downtown Manhattan


In [17]:
# 전체 뉴스 수집 + 이원화 분류 + 감성 분석 한번에
TICKER_QUERIES = {
    "AAPL":  "Apple Inc stock market",
    "MSFT":  "Microsoft stock market",
    "GOOGL": "Google Alphabet stock market",
    "JPM":   "JPMorgan Chase stock market",
    "SOXX": "SOXX stock market"
}

# 거시경제 뉴스도 별도 수집
SYSTEMIC_QUERY = "federal reserve inflation recession tariff economy 2026"

def fetch_and_classify(query, api_key, days_back=30):
    """뉴스 수집 + 분류 + 감성 분석 한번에"""
    from datetime import datetime, timedelta

    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days_back)).strftime("%Y-%m-%d")

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "from": start_date,
        "to": end_date,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 50,
        "apiKey": api_key
    }

    response = requests.get(url, params=params)
    data = response.json()

    if data["status"] != "ok":
        print(f"에러: {data.get('message')}")
        return []

    results = []
    for article in data["articles"]:
        headline = article["title"]
        category = classify_news(headline)
        sentiment = nlp(headline[:512])[0]
        score_map = {"positive": 1, "negative": -1, "neutral": 0}
        sentiment_score = score_map[sentiment["label"]] * sentiment["score"]

        results.append({
            "headline": headline,
            "category": category,
            "sentiment_label": sentiment["label"],
            "sentiment_score": round(sentiment_score, 4),
            "published_at": article["publishedAt"]
        })

    return results

# 거시경제 뉴스 수집
print("거시경제 뉴스 수집 중...")
systemic_news = fetch_and_classify(SYSTEMIC_QUERY, NEWSAPI_KEY)
print(f"  수집: {len(systemic_news)}개")

# 종목별 뉴스 수집
all_news = {}
for ticker, query in TICKER_QUERIES.items():
    print(f"{ticker} 뉴스 수집 중...")
    all_news[ticker] = fetch_and_classify(query, NEWSAPI_KEY)
    print(f"  수집: {len(all_news[ticker])}개")

print("\n완료!")

거시경제 뉴스 수집 중...
  수집: 3개
AAPL 뉴스 수집 중...
  수집: 49개
MSFT 뉴스 수집 중...
  수집: 50개
GOOGL 뉴스 수집 중...
  수집: 50개
JPM 뉴스 수집 중...
  수집: 49개
SOXX 뉴스 수집 중...
  수집: 50개

완료!


In [12]:
# 거시경제 뉴스 3개는 너무 적음 → 검색어 분리해서 더 수집
SYSTEMIC_QUERIES = [
    "federal reserve interest rate 2026",
    "inflation economy recession 2026",
    "tariff trade war market 2026"
]

systemic_news = []
for query in SYSTEMIC_QUERIES:
    news = fetch_and_classify(query, NEWSAPI_KEY)
    systemic_news.extend(news)
    print(f"  '{query}': {len(news)}개")

# 중복 제거
systemic_df = pd.DataFrame(systemic_news).drop_duplicates(subset=["headline"])
print(f"\n거시경제 뉴스 총합 (중복 제거): {len(systemic_df)}개")
systemic_df.head()

  'federal reserve interest rate 2026': 50개
  'inflation economy recession 2026': 49개
  'tariff trade war market 2026': 50개

거시경제 뉴스 총합 (중복 제거): 145개


,headline,category,sentiment_label,sentiment_score,published_at
0,Markets price in 21% chance of Fed rate cut in...,systemic,positive,0.6783,2026-07-07T06:20:13Z
1,Markets price in just 21% chance of Fed rate c...,systemic,positive,0.4277,2026-07-07T06:06:35Z
2,Polymarket traders see 79% chance Fed will not...,systemic,negative,-0.7986,2026-07-07T05:07:03Z
3,BHP Shares Fall Nearly 2% as Iron Ore Prices a...,unrelated,negative,-0.9708,2026-07-07T04:45:38Z
4,Gold pulls back from two-week high on stronger...,systemic,positive,0.5306,2026-07-07T03:57:23Z


In [19]:
# 종목별 뉴스도 DataFrame으로 정리
idiosyncratic_dfs = {}
for ticker, news in all_news.items():
    df = pd.DataFrame(news).drop_duplicates(subset=["headline"])
    # idiosyncratic 카테고리만 필터링
    idio_df = df[df["category"] == f"idiosyncratic_{ticker}"]
    idiosyncratic_dfs[ticker] = idio_df
    print(f"{ticker}: 전체 {len(df)}개 → 고유 뉴스 {len(idio_df)}개")

# 거시경제 뉴스만 필터링
systemic_only = systemic_df[systemic_df["category"] == "systemic"]
print(f"\nSystemic 뉴스: {len(systemic_only)}개")

AAPL: 전체 47개 → 고유 뉴스 10개
MSFT: 전체 50개 → 고유 뉴스 11개
GOOGL: 전체 48개 → 고유 뉴스 19개
JPM: 전체 47개 → 고유 뉴스 2개
SOXX: 전체 49개 → 고유 뉴스 7개

Systemic 뉴스: 40개


In [20]:
# 이원화된 감성 점수 계산
def get_sentiment_score(df):
    """DataFrame에서 평균 감성 점수 계산"""
    if df.empty:
        return 0.0
    return round(df["sentiment_score"].mean(), 4)

# 거시경제 감성 점수 (HMM 국면 조정에 사용)
systemic_score = get_sentiment_score(systemic_only)
print(f"Systemic 감성 점수: {systemic_score:.4f}")
print(f"  → {'부정적' if systemic_score < 0 else '긍정적'} 거시환경\n")

# 종목별 고유 감성 점수 (개별 리스크 가중치에 사용)
idiosyncratic_scores = {}
for ticker, df in idiosyncratic_dfs.items():
    score = get_sentiment_score(df)
    idiosyncratic_scores[ticker] = score
    print(f"{ticker} 고유 감성 점수: {score:.4f} ({'부정적' if score < 0 else '긍정적'})")

# 저장
today = pd.Timestamp.today().strftime("%Y-%m-%d")

systemic_df_save = pd.DataFrame([{
    "date": today,
    "systemic_score": systemic_score
}])

idio_df_save = pd.DataFrame([{
    "date": today,
    "ticker": ticker,
    "idiosyncratic_score": score
} for ticker, score in idiosyncratic_scores.items()])

systemic_df_save.to_csv(f"{data_path}/sentiment_systemic.csv", index=False)
idio_df_save.to_csv(f"{data_path}/sentiment_idiosyncratic.csv", index=False)

print("\n저장 완료:")
print("  sentiment_systemic.csv")
print("  sentiment_idiosyncratic.csv")

Systemic 감성 점수: -0.1534
  → 부정적 거시환경

AAPL 고유 감성 점수: -0.1298 (부정적)
MSFT 고유 감성 점수: -0.2807 (부정적)
GOOGL 고유 감성 점수: -0.0206 (부정적)
JPM 고유 감성 점수: -0.3672 (부정적)
SOXX 고유 감성 점수: 0.0671 (긍정적)

저장 완료:
  sentiment_systemic.csv
  sentiment_idiosyncratic.csv
